# Six-Day Public-State Fieldbook

This is a small public-state router around strong public action histories. It
does not try to invent all 720 actions online. Instead, it follows a reviewed
six-day plan, looks at the farm and market, and chooses the next six-day plan.

## What it does

There are 24 turns per day. The season is split into five 144-turn blocks:

| period | turn | available plans | decision |
|---|---:|---:|---|
| day 0–6 | 0–143 | 1 | fixed opening |
| day 6–12 | 144–287 | 2 | public-state tree |
| day 12–18 | 288–431 | 1 | fixed bridge |
| day 18–24 | 432–575 | 3 | public-state tree |
| day 24–30 | 576–719 | 5 | public-state tree |

At a boundary, the router uses only information already available in the
observation: unlocked shops, market prices and inventory, public farm state,
and its own private shed and hand inventories. It never keys on a player name,
leaderboard rank, replay id, random seed, final result, or a future shop.

## How I built it

1. I downloaded the current top 200 and older public replays, then deduplicated
   complete action histories by exact SHA-256.
2. I collected every available public episode for rank 3 Bantam (83) and rank
   8 Zenith Ye (119), and treated their six-day blocks as swap candidates.
3. I tested candidate blocks at the same boundary, on fresh seeds and both
   seats, rather than comparing unrelated full games.
4. I retained eight distinct routes and fitted small decision trees using only
   public runtime features.
5. I froze the policy and evaluated it against 188 external histories from the
   current top-200 snapshot.

The search also used 400 historical top-200 episodes (363 unique complete
histories). Historical results were used to propose routes; fresh seed blocks
were kept for the final measurements.

## Four decisions, eight routes, thirty possible paths

The agent can switch at turns 144, 288, 432, and 576. Across the four deployed
trees there are 16 binary tests and 20 leaves. The route counts are
`1 × 2 × 1 × 3 × 5`, so there are 30 possible full-season route sequences.
Only eight route tapes are stored because the same base route is reused across
several blocks.

Typical tests ask questions such as “is carrot expensive?”, “is fertilizer
cheap?”, or “is the cash gap positive?”. They are ordinary comparisons against
the current observation, not opponent identification.

## Budget guard

Before starting a new block, the agent calculates the cost of purchases already
written into that block: hires, land, seeds, feed, fertilizer, and animals. If
cash is sufficient, it changes nothing. If cash is short, it protects the
block's required stock and sells only excess products, highest current price
first. This avoids blindly liquidating useful feed or fertilizer just because a
boundary was reached.

## Evidence and limits

On the frozen release block—188 opponents, 64 fresh seeds, both seats—the agent
recorded 22,795 wins, 58 draws, and 1,211 losses in 24,064 games: a 0.9485 point
rate with seed-bootstrap 95% CI `[0.9371, 0.9565]`. Across two fresh blocks it
recorded a 0.9503 point rate over 36,096 games. There were no runtime or schema
errors.

This is local evidence, not a leaderboard guarantee. The regular 164-opponent
subset scored 0.9948, while the deliberately hard 24-opponent subset scored
0.6322. The hard tail remains the main place to improve.

## Readable notebook, separated tapes

The long C++ action tables are intentionally not printed in this Notebook.
They live in the attached public source Dataset with hashes and an Apache-2.0
license. The cells below verify every source file, compile the Linux agent, and
leave `main.py`, `agent.so`, and a top-level two-file submission archive in
`/kaggle/working`.


## Build the reviewed agent

The long action tapes and C++ implementation live in the attached public **Six-Day Public-State Agent Source** Dataset. Keeping them there makes this Notebook short enough to read while preserving a hash-verifiable build.


In [ ]:
import hashlib
import shutil
from pathlib import Path

source = Path("/kaggle/input/six-day-public-state-agent-source")
root = Path("/kaggle/working/sixday_r4_source")
root.mkdir(parents=True, exist_ok=True)
expected = {'agent_main.py': '5809846fd528f00d1c01d6f58940633604adb7283d1e2f152603697967e3bb89', 'policy.cpp': 'a3f04707615b7c04f3ab4a3c593081c98df2dd091be386aa80a3ffad7c00c1b4', 'submission_bridge.cpp': '2e58be38b2f67e9ed2cfaa97d0534285b2af21c7d4e2be92735e647d4a68a4c5', 'policy_plugin_abi.hpp': '67679edc55b1951deac859d1191a6eb2198868b3b7237beb2266db9788d80345', 'pyrandom.hpp': '3835fe39fbc74b05078022a839dea1c1aff7a9d6d21a1227d4d6cd61a3612d75', 'sim.hpp': 'f483219c22dd9a05c00e28a200aa48d70d4a461915c09b3a0fb36c0330cc554c', 'six_day_budget_guard.hpp': '5cf24ed4099e5b19e9f195869881548f2145a40329fc30d11e682501c3688556'}
for name, digest in expected.items():
    incoming = source / name
    assert hashlib.sha256(incoming.read_bytes()).hexdigest() == digest
    shutil.copyfile(incoming, root / name)
print(f"Verified and staged {len(expected)} separated source files")


In [ ]:
import shutil
import subprocess
import tarfile

subprocess.run(
    [
        "g++", "-O3", "-std=c++17", "-shared", "-fPIC",
        "-I.", "-o", "/kaggle/working/agent.so",
        "policy.cpp", "submission_bridge.cpp",
    ],
    cwd=root,
    check=True,
)
shutil.copyfile(root / "agent_main.py", "/kaggle/working/main.py")
archive = "/kaggle/working/sixday-publicstate-agent.tar.gz"
with tarfile.open(archive, "w:gz") as bundle:
    bundle.add("/kaggle/working/main.py", arcname="main.py")
    bundle.add("/kaggle/working/agent.so", arcname="agent.so")
print("Built main.py, agent.so, and the top-level submission archive")


In [ ]:
import hashlib
import json
from pathlib import Path

def file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

print(json.dumps({
    "agent": 'hybrid_sixday_combined_publicstate_r4',
    "policy_sha256": 'a1ee929e5639cffbbf9c7722af147a674f1a973613c8fd0ac4f74bdad07c9355',
    "main_py_sha256": file_hash("/kaggle/working/main.py"),
    "agent_so_sha256": file_hash("/kaggle/working/agent.so"),
    "submission_archive_sha256": file_hash("/kaggle/working/sixday-publicstate-agent.tar.gz"),
    "runtime_features_exclude": ["opponent identity", "seed", "result", "future shops"],
}, indent=2))
